In [3]:
import requests
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [4]:
OUTPUT_DIR = "../Data"

### Monthly Data

In [5]:
weather_daily = pd.read_csv('../data/JC/jersey_weather_2025.csv')
citibike_df = pd.read_csv('../data/JC/JC2025_Enriched.csv')

In [6]:
weather_daily['date'] = pd.to_datetime(weather_daily['date'], errors="coerce")
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors="coerce")

In [7]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 998304 entries, 0 to 998303
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ride_id             998304 non-null  str           
 1   rideable_type       998304 non-null  str           
 2   started_at          998304 non-null  str           
 3   ended_at            998304 non-null  str           
 4   start_station_name  998304 non-null  str           
 5   start_station_id    998304 non-null  str           
 6   end_station_name    998304 non-null  str           
 7   end_station_id      998304 non-null  str           
 8   start_lat           998304 non-null  float64       
 9   start_lng           998304 non-null  float64       
 10  end_lat             998304 non-null  float64       
 11  end_lng             998304 non-null  float64       
 12  member_casual       998304 non-null  str           
 13  ride_duration_min   998304 non-null  flo

In [8]:
citibike_df.head()


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,month_number,day_of_week,hour,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,...,-74.051789,member,6.192900,2025-01-16,2025-01,January,1,Thursday,17,Winter
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,...,-74.078900,member,11.461350,2025-01-31,2025-01,January,1,Friday,6,Winter
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,21.377617,2025-01-09,2025-01,January,1,Thursday,16,Winter
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,22.934333,2025-01-21,2025-01,January,1,Tuesday,16,Winter
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,25.822100,2025-01-30,2025-01,January,1,Thursday,16,Winter


In [9]:
monthly_rides = (citibike_df
                 .groupby('month',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50478
2,2025-02,45131
3,2025-03,73124
4,2025-04,81295
5,2025-05,92882
6,2025-06,96738
7,2025-07,107376
8,2025-08,108003
9,2025-09,115581


In [10]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

### Number of Rides per Season

In [11]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,143477
1,Spring,247301
2,Summer,312117
0,Autumn,295409


In [12]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

### Merge with Weather Data

In [13]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,2
1,2025-01-01,1174
2,2025-01-02,1709
3,2025-01-03,1765
4,2025-01-04,1336


In [14]:
weather_daily.head()

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-01,10.9,3.9,7.4,4.5,4.5,0.0,23.2
1,2025-01-02,5.4,0.3,2.6,0.0,0.0,0.0,25.1
2,2025-01-03,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
3,2025-01-04,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1
4,2025-01-05,0.3,-3.6,-2.2,0.0,0.0,0.0,19.9


In [15]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1174,10.9,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1709,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1765,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1336,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [16]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

### Rides vs Average Temperature

In [19]:
import statsmodels.api as sm

fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Wind Speed

In [21]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Precipitation

In [23]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

### Rides vs Temperature

In [24]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)


In [25]:
fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

### Number of Rides by DOW

In [27]:
days = (citibike_df
.groupby("day_of_week", as_index=False)
.agg(number_of_rides=("ride_id", "count"))
)

In [28]:
day_order = [
"Monday",
"Tuesday",
"Wednesday",
"Thursday",
"Friday",
"Saturday",
"Sunday"
]

In [29]:
days["day_of_week"] = pd.Categorical(
days["day_of_week"],
categories=day_order,
ordered=True
)

days = days.sort_values("day_of_week")

In [30]:
days["Type"] = "Normal"

days.loc[
days["number_of_rides"].idxmax(),
"Type"
] = "Highest"

days.loc[
days["number_of_rides"].idxmin(),
"Type"
] = "Lowest"

In [31]:
fig = px.bar(
days,
x="day_of_week",
y="number_of_rides",
color="Type",
text_auto=True,
title="Number of Citi Bike Rides by Day of Week"
)

fig.update_layout(
xaxis_title="Day of Week",
yaxis_title="Number of Rides"
)

fig.show()

### Number of Rides by Hour

In [33]:
hourly = (
citibike_df
.groupby("hour", as_index=False)
.agg(number_of_rides=("ride_id","count"))
)

In [34]:
fig = px.line(
hourly,
x="hour",
y="number_of_rides",
markers=True,
title="Hourly Citi Bike Demand"
)

fig.update_layout(
xaxis_title="Hour of Day",
yaxis_title="Number of Rides"
)

fig.show()

### Top 10 Start Stations

In [42]:
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,44986
58,Hoboken Terminal - Hudson St & Hudson Pl,25879
53,Hamilton Park,22232
95,River St & Newark St,21384
86,Newport PATH,20641
18,Bergen Ave & Sip Ave,20370
44,Exchange Pl,19983
0,11 St & Washington St,19470
94,River St & 1 St,19127
87,Newport Pkwy,18709


In [43]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

### Top 15 End Stations

In [45]:
N = 15
top_stations = (citibike_df
                .groupby('end_station_name', as_index=False)
                .agg(number_of_arrivals = ('ride_id','count') )
                .sort_values("number_of_arrivals",ascending=False)
                .head(N))

top_stations

,end_station_name,number_of_arrivals
232,Grove St PATH,47744
241,Hoboken Terminal - Hudson St & Hudson Pl,26638
233,Hamilton Park,22347
347,River St & Newark St,22113
317,Newport PATH,20698
73,Bergen Ave & Sip Ave,20357
207,Exchange Pl,20144
7,11 St & Washington St,19501
318,Newport Pkwy,18705
346,River St & 1 St,18515


In [46]:
fig = px.bar(
    data_frame=top_stations,
    x = 'number_of_arrivals',
    y = 'end_station_name',
    orientation = 'h',
    title=f"Top {N} Stations by Number of Departures",
    text_auto= True

)
fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)


fig.show()

### Concatinating the stations and visualization

In [48]:
N = 15

start = (
citibike_df
.groupby("start_station_name", as_index=False)
.agg(number_of_rides=("ride_id","count"))
.rename(columns={"start_station_name":"station"})
)

start["Type"] = "Departures"

In [49]:
end = (
citibike_df
.groupby("end_station_name", as_index=False)
.agg(number_of_rides=("ride_id","count"))
.rename(columns={"end_station_name":"station"})
)

end["Type"] = "Arrivals"

In [50]:
stations = pd.concat([start, end])


In [51]:
stations = (
stations
.sort_values("number_of_rides", ascending=False)
.head(N)
)

In [52]:
fig = px.bar(
stations,
x="number_of_rides",
y="station",
color="Type",
orientation="h",
barmode="group",
text_auto=True,
title=f"Top {N} Citi Bike Stations: Departures vs Arrivals"
)

fig.update_layout(
xaxis_title="Number of Rides",
yaxis_title="Station"
)

fig.show()